# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step workflow for loading, exploring, and processing the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their field and column `@id` references.

We will display all record sets and for each, the field `@id`s and column `@id`s as declared in the Croissant metadata.

In [ ]:
# List all available record sets by @id and show their fields and columns
record_sets = metadata.record_sets
print(f"Found {len(record_sets)} record sets.\n")
for rs in record_sets:
    print(f"Record Set: {rs.id}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.id} (column @id: {field.column.id if getattr(field, 'column', None) else 'N/A'})")
    else:
        print("  No fields listed.")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Entities are always referenced by their `@id`, following Croissant convention.

Below we extract all record sets as DataFrames. Choose relevant ones for further analysis.

In [ ]:
# Extract data from each record set (example shown for all)
from pprint import pprint

dataframes = {}
for rs in metadata.record_sets:
    # Use the record set @id for loading
    rs_id = rs.id
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from record set '@id': {rs_id}")
        if not df.empty:
            print(f"Columns: {df.columns.tolist()}")
        print()
    except Exception as e:
        print(f"Could not load records for record set {rs_id}: {e}")
        print()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data for further analysis.

Below we select the main protein record set (the one describing protein entries, typically the largest), and:
- Filter proteins for those with a molecular weight greater than a threshold.
- Normalize the 'molecular_weight' field.
- Group by a categorical field such as 'sample' or 'accession' as appropriate.

You can modify the `mw_field_id` and `group_field_id` variables to refer to the correct field `@id`s from above.


In [ ]:
# Choose a record set by its @id for EDA
main_record_set_id = None
for rs_id, df in dataframes.items():
    if not df.empty and 'molecular_weight' in df.columns:
        main_record_set_id = rs_id
        break

if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Use @id for fields as required (update as needed)
    mw_field_id = 'molecular_weight'  # Replace with actual field @id from above if necessary
    # Example: 'accession' or any categorical/identifier field for grouping
    group_field_id = 'accession' if 'accession' in df.columns else df.columns[0]

    # Filter proteins with molecular weight > 30000
    threshold = 30000
    filtered_df = df[df[mw_field_id].astype(float) > threshold]
    print(f"Filtered records with {mw_field_id} > {threshold} (count={len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize the molecular weight
    filtered_df = filtered_df.copy()  # avoid pandas warning
    filtered_df[f"{mw_field_id}_normalized"] = (filtered_df[mw_field_id].astype(float) - filtered_df[mw_field_id].astype(float).mean()) / filtered_df[mw_field_id].astype(float).std()
    print(f"Normalized {mw_field_id} for filtered records:")
    display(filtered_df[[mw_field_id, f"{mw_field_id}_normalized"]].head())

    # Group by categorical field (e.g. accession)
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by '{group_field_id}':")
        display(grouped_df.head())
else:
    print('No record set with `molecular_weight` field found.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of molecular weights, and a scatter plot (if both abundance and molecular weight are available) for a quick look at possible relationships.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None:
    # Histogram of molecular weights
    plt.figure(figsize=(8,4))
    sns.histplot(df[mw_field_id].astype(float), bins=30, kde=True)
    plt.title('Distribution of Molecular Weight')
    plt.xlabel('Molecular Weight (Da)')
    plt.ylabel('Count')
    plt.show()

    # Example scatter plot: molecular_weight vs. abundance if available
    abundance_candidates = [c for c in df.columns if 'abundance' in c]
    if abundance_candidates:
        abundance_field = abundance_candidates[0]
        plt.figure(figsize=(8,6))
        sns.scatterplot(
            x=df[mw_field_id].astype(float),
            y=df[abundance_field].astype(float),
            alpha=0.5
        )
        plt.xlabel('Molecular Weight (Da)')
        plt.ylabel(abundance_field)
        plt.title('Molecular Weight vs Abundance')
        plt.show()
else:
    print('No main protein record set available for visualization.')

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to:
- Discover and enumerate the available record sets, fields, and columns in the FAIR^2 dataset using their Croissant `@id`s
- Load and explore records in `pandas` DataFrames
- Perform filtering, normalization, and grouping analyses on protein records
- Visualize the distribution of molecular weights and examine abundance relationships

For more advanced analyses, refer to the `mlcroissant` [documentation](https://mlcommons.github.io/croissant/api/) and the SenScience FAIR^2 [dataset portal](https://sen.science/doi/10.71728/senscience.vq4a-28xa).
